# Lab 12: Android Malware Detection with a Random Forest
Course 314341 AI in Cybersecurity and Cloud Computing, Ajloun National University. Dr. Mohammed Tawfik.

Run the cells from top to bottom in Google Colab (Runtime → Run all, or Shift+Enter cell by cell). Read the comments: lines that start with # explain the code.

**Data:** `drebin-215-dataset-5560malware-9476-benign.csv` from Kaggle [Android Malware Dataset for Machine Learning](https://www.kaggle.com/datasets/shashwatwork/android-malware-dataset-for-machine-learning): 15,036 apps, 215 yes/no features (permissions and API calls), column `class` = malware or benign. Only features, no real malware: safe to open.

## 1. Load and prepare

In [ ]:
import pandas as pd, numpy as np
data = pd.read_csv("drebin-215-dataset-5560malware-9476-benign.csv", low_memory=False)
print(data.shape)
print(data["class"].value_counts())        # the class with 5,560 rows is malware
data = data.replace("?", np.nan).dropna()  # a few cells contain "?"
X = data.drop(columns=["class"]).astype(int)
y = data["class"]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 2. One decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
tree = DecisionTreeClassifier(max_depth=6, random_state=42).fit(X_train, y_train)
print(classification_report(y_test, tree.predict(X_test)))

## 3. A random forest: 100 trees that vote

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
p = rf.predict(X_test)
print(confusion_matrix(y_test, p))
print(classification_report(y_test, p))

## 4. Which permissions and API calls matter most?

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(15))
imp.head(15)[::-1].plot(kind="barh", figsize=(7, 5), title="Most important features")